# Distributional enrichment, corrected

The first attempt compared the current baseline against `uci_mean_SD_enrichment.csv`, which turned out to be an **older cohort**: 100,911 rows and 154 features, built before the expired/hospice, paediatric and unknown-sex exclusions. The apparent +0.0052 gain was a population difference, not an enrichment effect.

This notebook fixes that. The dispersion columns take only 23–24 distinct values, one per demographic stratum, so they can be recovered as a lookup keyed on (age band, sex, race) and joined onto the **current** cohort. The comparison then holds the cohort fixed and varies only the added columns.

**Before you run:** Runtime → Change runtime type → **T4 GPU**. Then Run all. About 7 minutes.

In [ ]:
#@title 1. Environment
!pip install -q xgboost==3.2.0 scikit-learn==1.8.0 imbalanced-learn==0.14.1 2>/dev/null
import numpy as np, pandas as pd, sklearn, xgboost as xgb, warnings, os, glob
warnings.filterwarnings('ignore')
try:
    d0=xgb.DMatrix(np.zeros((16,3)),label=np.array([0,1]*8))
    xgb.train({'tree_method':'hist','device':'cuda'},d0,num_boost_round=1); USE_GPU=True
except Exception: USE_GPU=False
print('sklearn',sklearn.__version__,'| xgboost',xgb.__version__,'| GPU',USE_GPU)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
TARGET,GROUP,N_SPLITS='readmitted','patient_nbr',5
CAT_AS_STR=['admission_type_id','discharge_disposition_id','admission_source_id','race']
SEED=42
def mk(pw,seed=SEED):
    kw=dict(n_estimators=500,max_depth=4,learning_rate=0.03,subsample=0.8,colsample_bytree=0.7,
            min_child_weight=5,reg_lambda=2.0,scale_pos_weight=pw,eval_metric='logloss',
            random_state=seed,tree_method='hist')
    if USE_GPU: kw['device']='cuda'
    return XGBClassifier(**kw)
def prep(df):
    df=df.copy()
    if df[TARGET].dtype==object: df[TARGET]=(df[TARGET].astype(str)=='<30').astype(int)
    y=df[TARGET].astype(int).values; g=df[GROUP].values
    X=df.drop(columns=[TARGET,GROUP])
    for c in CAT_AS_STR:
        if c in X.columns: X[c]=X[c].astype(str)
    cat=[c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    return pd.get_dummies(X,columns=cat,dummy_na=False).astype(float),y,g
def oof_auc(X,y,g,seed=SEED):
    Xv=X.values.astype(float); pw=(y==0).sum()/max((y==1).sum(),1)
    oof=np.zeros(len(y))
    for tr,te in StratifiedGroupKFold(N_SPLITS,shuffle=True,random_state=seed).split(np.zeros(len(y)),y,g):
        sc=StandardScaler().fit(Xv[tr]); m=mk(pw,seed); m.fit(sc.transform(Xv[tr]),y[tr])
        oof[te]=m.predict_proba(sc.transform(Xv[te]))[:,1]
    return roc_auc_score(y,oof),oof

In [ ]:
#@title 2. Load the current cohort and the stale mean+SD file
from google.colab import drive
drive.mount('/content/drive')
def find(n):
    h=glob.glob(f'/content/drive/**/{n}',recursive=True); return h[0] if h else None
P_BASE=find('uci_baseline.csv'); P_MEAN=find('uci_mean_clinical.csv'); P_MSD=find('uci_mean_SD_enrichment.csv')
print('baseline :',P_BASE); print('mean     :',P_MEAN); print('mean+SD  :',P_MSD)

base=pd.read_csv(P_BASE,low_memory=False)
msd =pd.read_csv(P_MSD ,low_memory=False)
if base[TARGET].dtype==object: base[TARGET]=(base[TARGET].astype(str)=='<30').astype(int)
print(f'\ncurrent cohort : {len(base)} rows, {int(base[TARGET].sum())} events')
print(f'stale mean+SD  : {len(msd)} rows  <- different cohort, used only as a lookup source')

In [ ]:
#@title 3. Recover the dispersion lookup keyed on (age, sex, race)
SD_COLS=[c for c in msd.columns if c.endswith('_SD')]
KEY=['age','gender','race']
print('dispersion columns:',SD_COLS)

# one row per stratum; verify the value is constant within stratum before using it
chk=msd.groupby(KEY)[SD_COLS].nunique()
bad=(chk>1).sum().sum()
print('strata in lookup source:',len(chk))
print('cells where the SD is NOT constant within stratum:',int(bad))
assert bad==0, 'dispersion is not a deterministic function of (age, sex, race) - stop and inspect'
print('confirmed: each dispersion value is a deterministic function of the demographic key')

lookup=msd.groupby(KEY,as_index=False)[SD_COLS].first()
print('\nlookup table:',lookup.shape)
display(lookup.head(8))

In [ ]:
#@title 4. Join the dispersion columns onto the CURRENT cohort
enr_sd=base.merge(lookup,on=KEY,how='left')
assert len(enr_sd)==len(base), 'merge changed the row count'
miss=enr_sd[SD_COLS].isna().sum()
print('rows after merge:',len(enr_sd),'(cohort preserved)')
print('missing values per dispersion column after join:')
print(miss.to_string())
if miss.sum()>0:
    # a stratum present in the cohort but absent from the lookup falls back to the column mean
    for c in SD_COLS: enr_sd[c]=enr_sd[c].fillna(lookup[c].mean())
    print('\nunmatched strata filled with the lookup mean')
print('\ndistinct values per dispersion column on the current cohort:')
for c in SD_COLS: print(f'  {c}: {enr_sd[c].nunique()}')

In [ ]:
#@title 5. Four configurations, cohort held fixed
mean_df=pd.read_csv(P_MEAN,low_memory=False)
if mean_df[TARGET].dtype==object: mean_df[TARGET]=(mean_df[TARGET].astype(str)=='<30').astype(int)

# mean + SD on the current cohort: start from the mean-enriched file, add the SD columns
mean_sd_df=mean_df.merge(lookup,on=KEY,how='left')
for c in SD_COLS: mean_sd_df[c]=mean_sd_df[c].fillna(lookup[c].mean())

CONFIGS=[('baseline',base),('SD only',enr_sd),('mean only',mean_df),('mean + SD',mean_sd_df)]
res={}; oofs={}
for nm,df in CONFIGS:
    X,y,g=prep(df)
    a,o=oof_auc(X,y,g); res[nm]=dict(auroc=a,f=X.shape[1],n=len(y),ev=int(y.sum())); oofs[nm]=(o,y,g)
    print(f'{nm:12s} AUROC={a:.4f}  features={X.shape[1]:4d}  n={len(y)}  events={int(y.sum())}',flush=True)

ns={d['n'] for d in res.values()}
print('\ncohort identical across all configurations:', len(ns)==1, ns)
b=res['baseline']['auroc']
print()
for nm in ['SD only','mean only','mean + SD']:
    print(f'  {nm:12s} minus baseline: {res[nm]["auroc"]-b:+.4f}')

In [ ]:
#@title 6. Paired patient-clustered bootstrap against baseline
N_BOOT=1000  #@param {type:"integer"}
ob,yb,gb=oofs['baseline']
rng=np.random.default_rng(SEED)
pats=np.unique(gb); by={}
for p_,i_ in zip(gb,np.arange(len(gb))): by.setdefault(p_,[]).append(i_)
by={k:np.array(v) for k,v in by.items()}
boot_idx=[]
for _ in range(N_BOOT):
    pick=rng.choice(pats,size=len(pats),replace=True)
    boot_idx.append(np.concatenate([by[p_] for p_ in pick]))

rows=[]
for nm in ['SD only','mean only','mean + SD']:
    oc,yc,_=oofs[nm]
    diffs=[]
    for idx in boot_idx:
        if len(np.unique(yb[idx]))<2: continue
        diffs.append(roc_auc_score(yb[idx],oc[idx])-roc_auc_score(yb[idx],ob[idx]))
    diffs=np.array(diffs)
    obs=res[nm]['auroc']-res['baseline']['auroc']
    lo,hi=np.percentile(diffs,[2.5,97.5])
    p=2*min((diffs<=0).mean(),(diffs>=0).mean()); p=min(max(p,1/len(diffs)),1.0)
    rows.append(dict(configuration=nm,delta_auroc=round(obs,4),ci_low=round(lo,4),ci_high=round(hi,4),p=round(p,3)))
cmp=pd.DataFrame(rows); display(cmp)

In [ ]:
#@title 7. Summary
lines=[]
def log(s): print(s); lines.append(s)
log(f'environment: xgboost {xgb.__version__}, sklearn {sklearn.__version__}, GPU {USE_GPU}')
log('')
log('DISTRIBUTIONAL ENRICHMENT, COHORT HELD FIXED')
for nm,d in res.items():
    log(f'  {nm:12s} AUROC {d["auroc"]:.4f}  ({d["f"]} features, n={d["n"]}, events={d["ev"]})')
log('')
log(cmp.to_string(index=False))
log('')
log(f'dispersion columns ({len(SD_COLS)}): {SD_COLS}')
log('distinct values per dispersion column on the current cohort:')
for c in SD_COLS: log(f'  {c}: {enr_sd[c].nunique()}')
log('')
log('NOTE: only three measures have dispersion available (BMI, systolic blood pressure,')
log('glycated haemoglobin), so this is a partial test of distributional enrichment.')
open('distributional_enrichment_fixed.txt','w').write('\n'.join(lines))
from google.colab import files; files.download('distributional_enrichment_fixed.txt')